In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Enhanced Drought Outlook Analysis with Multi-Core Processing

This script calculates drought outlook values using historical CDI data,
creates a NetCDF file, and generates an interactive dashboard with:
1. Month-pair consistency maps
2. Monthly outlook maps with percentage-based coloring
3. Yearly map collections (each year in its own folder)
4. Interactive HTML dashboard with Bootstrap UI and visualization options

Features:
- Multi-core processing for faster calculations
- Interactive dashboard with drought/no-drought toggle views
- Percentage-based color mapping
- Responsive Bootstrap UI with professional design
- Progress tracking and performance metrics

Author: Created based on specific requirements
Date: May 2025
"""

import os
import sys
import time
import logging
import numpy as np
import xarray as xr
import pandas as pd
import calendar
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import multiprocessing as mp
from functools import partial
import json
from tqdm import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("outlook_cdi.log")
    ]
)
logger = logging.getLogger(__name__)

# Create a separate history log file
history_log = open("calculation_history.log", "w")

def write_history(message):
    """Write a message to the history log file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    history_log.write(f"{timestamp} - {message}\n")
    history_log.flush()  # Ensure the message is written immediately

def load_original_cdi(file_path, end_year=2018):
    """
    Load the original CDI file and extract data up to the specified end year.
    
    Args:
        file_path (str): Path to the original CDI file
        end_year (int): End year for the data (inclusive)
        
    Returns:
        xarray.Dataset: The loaded CDI dataset filtered to the specified end year
    """
    logger.info(f"Step 1: Loading original CDI file from {file_path}")
    write_history(f"Loading original CDI file: {file_path}")
    
    ds = xr.open_dataset(file_path)
    
    # Log basic information
    logger.info(f"  Original dimensions: {ds.dims}")
    write_history(f"Original dimensions: {ds.dims}")
    
    # Convert time values to datetime for easier filtering
    times = pd.to_datetime(ds.time.values)
    logger.info(f"  Original time range: {times[0]} to {times[-1]}")
    write_history(f"Original time range: {times[0]} to {times[-1]}")
    write_history(f"Total time steps: {len(times)}")
    
    # Filter to include only data up to the specified end year
    time_mask = times.year <= end_year
    filtered_times = times[time_mask]
    
    logger.info(f"  Filtering to data up to {end_year}")
    logger.info(f"  Filtered time range: {filtered_times[0]} to {filtered_times[-1]}")
    logger.info(f"  Filtered time steps: {len(filtered_times)}")
    write_history(f"Filtering to data up to {end_year}")
    write_history(f"Filtered time range: {filtered_times[0]} to {filtered_times[-1]}")
    write_history(f"Filtered time steps: {len(filtered_times)}")
    
    # Create a new dataset with only the filtered time steps
    ds_filtered = ds.sel(time=ds.time[time_mask])
    
    return ds_filtered

def process_month_pair(month_pair, ds, times_by_month, times_dt, height, width, drought_threshold=0.3):
    """
    Process a single month pair for parallelization.
    
    Args:
        month_pair (tuple): Tuple of (month1, month2)
        ds (xarray.Dataset): Dataset with CDI data
        times_by_month (dict): Dictionary mapping months to years to time indices
        times_dt (list): List of datetime objects for all time steps
        height (int): Grid height
        width (int): Grid width
        drought_threshold (float): Threshold for drought classification
        
    Returns:
        tuple: (pair_name, pair_stats) - Pair name and statistics dictionary
    """
    m1, m2 = month_pair
    month1_name = calendar.month_name[m1]
    month2_name = calendar.month_name[m2]
    pair_name = f"{month1_name}-{month2_name}"
    
    # Determine if this is a cross-year pair (December-January)
    is_cross_year = (m1 == 12 and m2 == 1)
    
    # Determine the start year
    # For Apr-May through Dec-Jan, start from 1998
    # For Jan-Feb, Feb-Mar, Mar-Apr, start from 1999 (as Apr 1998 is the first month)
    start_year = 1999 if m1 < 4 and m2 <= 4 else 1998
    
    # Determine end year
    end_year = 2018
    if is_cross_year:
        # For Dec-Jan, we stop at Dec 2018 (can't go to Jan 2019)
        end_year = 2017
    
    years_to_process = list(range(start_year, end_year + 1))
    num_years = len(years_to_process)
    
    # Keep track of which years we successfully processed
    processed_years = []
    
    # Initialize arrays for this month pair
    consistency_sum = np.zeros((height, width), dtype=np.float32)
    valid_count = np.zeros((height, width), dtype=np.int32)
    
    # Additional arrays for tracking drought status
    drought_sum = np.zeros((height, width), dtype=np.float32)
    
    # Process each year
    for year in years_to_process:
        # Determine the two months to compare
        if is_cross_year:
            month1_year = year
            month2_year = year + 1
        else:
            month1_year = month2_year = year
        
        # Check if both months exist in our dataset
        if m1 in times_by_month and month1_year in times_by_month[m1] and \
           m2 in times_by_month and month2_year in times_by_month[m2]:
            
            # Get the time indices
            idx1 = times_by_month[m1][month1_year]
            idx2 = times_by_month[m2][month2_year]
            
            # Get the CDI values for both months
            cdi1 = ds.cdi.values[:, :, idx1]
            cdi2 = ds.cdi.values[:, :, idx2]
            
            # Create a mask for valid data (between 0 and 1, not NaN)
            valid_mask1 = ~np.isnan(cdi1) & (cdi1 >= 0) & (cdi1 <= 1)
            valid_mask2 = ~np.isnan(cdi2) & (cdi2 >= 0) & (cdi2 <= 1)
            valid_mask = valid_mask1 & valid_mask2
            
            # Determine drought status (only for valid data)
            drought1 = np.full(cdi1.shape, np.nan, dtype=np.float32)
            drought2 = np.full(cdi2.shape, np.nan, dtype=np.float32)
            
            drought1[valid_mask] = (cdi1[valid_mask] <= drought_threshold).astype(np.float32)
            drought2[valid_mask] = (cdi2[valid_mask] <= drought_threshold).astype(np.float32)
            
            # Calculate consistency (1 if same status, 0 if different)
            is_consistent = np.full(cdi1.shape, np.nan, dtype=np.float32)
            is_consistent[valid_mask] = (drought1[valid_mask] == drought2[valid_mask]).astype(np.float32)
            
            # Update consistency sum and count
            consistency_sum[valid_mask] += is_consistent[valid_mask]
            valid_count[valid_mask] += 1
            
            # Update drought sum (for calculating drought percentage)
            drought_sum[valid_mask] += drought1[valid_mask]
            
            processed_years.append(year)
    
    # Calculate the consistency percentage
    percentage = np.full((height, width), np.nan, dtype=np.float32)
    nonzero_mask = (valid_count > 0)
    percentage[nonzero_mask] = (consistency_sum[nonzero_mask] / valid_count[nonzero_mask]) * 100.0
    
    # Calculate the drought percentage
    drought_pct = np.full((height, width), np.nan, dtype=np.float32)
    drought_pct[nonzero_mask] = (drought_sum[nonzero_mask] / valid_count[nonzero_mask]) * 100.0
    
    # Store number of years processed
    actual_years = len(processed_years)
    
    # Calculate statistics for this month pair
    valid_percentages = percentage[~np.isnan(percentage)]
    valid_drought_pct = drought_pct[~np.isnan(drought_pct)]
    
    # Get the coordinates
    latitude = ds.latitude.values
    longitude = ds.longitude.values
    
    pair_stats = {
        'processed_years': processed_years,
        'num_years': actual_years,
        'min_percent': np.min(valid_percentages) if valid_percentages.size > 0 else np.nan,
        'max_percent': np.max(valid_percentages) if valid_percentages.size > 0 else np.nan,
        'mean_percent': np.mean(valid_percentages) if valid_percentages.size > 0 else np.nan,
        'min_drought_pct': np.min(valid_drought_pct) if valid_drought_pct.size > 0 else np.nan,
        'max_drought_pct': np.max(valid_drought_pct) if valid_drought_pct.size > 0 else np.nan,
        'mean_drought_pct': np.mean(valid_drought_pct) if valid_drought_pct.size > 0 else np.nan,
        'valid_cells': np.sum(nonzero_mask),
        'percentage_data': percentage,  # Store the actual percentage data for mapping
        'drought_pct_data': drought_pct,  # Store the drought percentage data
        'latitude': latitude,
        'longitude': longitude
    }
    
    return pair_name, pair_stats

def calculate_historical_consistency_parallel(ds, drought_threshold=0.3, num_processes=None):
    """
    Calculate historical drought consistency for each month pair using parallel processing.
    
    Args:
        ds (xarray.Dataset): Dataset with CDI data
        drought_threshold (float): Threshold for drought classification
        num_processes (int, optional): Number of processes to use. Default is None (use all available cores).
        
    Returns:
        tuple: (ds_outlook, stats_dict) - Output dataset and statistics dictionary
    """
    logger.info(f"Step 2: Calculating historical drought consistency in parallel (threshold={drought_threshold})")
    write_history(f"STEP 2: Calculating historical drought consistency with threshold={drought_threshold} using parallel processing")
    
    # Extract dimensions and coordinates
    height = ds.dims['latitude']
    width = ds.dims['longitude']
    time_steps = ds.dims['time']
    
    latitude = ds.latitude.values
    longitude = ds.longitude.values
    times = ds.time.values
    times_dt = pd.to_datetime(times)
    
    write_history(f"Grid dimensions: {height} x {width}")
    write_history(f"Time period: {times_dt[0]} to {times_dt[-1]}")
    
    # Define all consecutive month pairs
    month_pairs = [
        (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7),
        (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 1)
    ]
    
    # Create the output dataset
    ds_outlook = xr.Dataset(
        coords={
            'latitude': (['latitude'], latitude),
            'longitude': (['longitude'], longitude),
            'time': (['time'], times)
        },
        attrs={
            'description': 'Drought outlook CDI based on historical consistency',
            'created_date': datetime.now().strftime('%Y-%m-%d'),
            'drought_threshold': str(drought_threshold),
            'calculation_method': 'Historical consistency percentage by month pair',
            'time_range': f'{times_dt[0].strftime("%Y-%m")} to {times_dt[-1].strftime("%Y-%m")}'
        }
    )
    
    # Initialize arrays for outlook CDI values
    outlook_cdi = np.full((height, width, time_steps), np.nan, dtype=np.float32)
    drought_pct_values = np.full((height, width, time_steps), np.nan, dtype=np.float32)
    
    # Statistics for the summary
    stats = {
        'month_pair_stats': {},
        'monthly_details': {},
        'start_time': times_dt[0],
        'end_time': times_dt[-1]
    }
    
    # Group times by month
    times_by_month = {}
    for i, t in enumerate(times_dt):
        month = t.month
        year = t.year
        if month not in times_by_month:
            times_by_month[month] = {}
        times_by_month[month][year] = i
    
    # Set up multiprocessing
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    logger.info(f"Using {num_processes} processes for parallel computation")
    write_history(f"Using {num_processes} processes for parallel computation")
    
    # Create a partial function with fixed arguments
    process_func = partial(
        process_month_pair,
        ds=ds,
        times_by_month=times_by_month,
        times_dt=times_dt,
        height=height,
        width=width,
        drought_threshold=drought_threshold
    )
    
    # Process month pairs in parallel
    start_time = time.time()
    
    # Using the Pool context manager
    with mp.Pool(processes=num_processes) as pool:
        results = list(tqdm(
            pool.imap(process_func, month_pairs),
            total=len(month_pairs),
            desc="Processing month pairs"
        ))
    
    end_time = time.time()
    
    logger.info(f"Parallel processing completed in {end_time - start_time:.2f} seconds")
    write_history(f"Parallel processing completed in {end_time - start_time:.2f} seconds")
    
    # Process the results
    for pair_name, pair_stats in results:
        # Store stats
        stats['month_pair_stats'][pair_name] = pair_stats
        
        # Log statistics
        logger.info(f"  Processed {pair_stats['num_years']} years for {pair_name}")
        logger.info(f"  Consistency percentage range: {pair_stats['min_percent']:.2f}% - {pair_stats['max_percent']:.2f}%")
        logger.info(f"  Drought percentage range: {pair_stats['min_drought_pct']:.2f}% - {pair_stats['max_drought_pct']:.2f}%")
        write_history(f"Statistics for {pair_name}:")
        write_history(f"  Consistency - Min: {pair_stats['min_percent']:.2f}%, Max: {pair_stats['max_percent']:.2f}%, Mean: {pair_stats['mean_percent']:.2f}%")
        write_history(f"  Drought - Min: {pair_stats['min_drought_pct']:.2f}%, Max: {pair_stats['max_drought_pct']:.2f}%, Mean: {pair_stats['mean_drought_pct']:.2f}%")
        
        # Map the historical percentage to each month in our time series
        month1, month2 = pair_name.split('-')
        m1 = list(calendar.month_name).index(month1)
        
        for i, t in enumerate(times_dt):
            if t.month == m1:
                # Store the historical percentage for this month
                outlook_cdi[:, :, i] = pair_stats['percentage_data']
                drought_pct_values[:, :, i] = pair_stats['drought_pct_data']
    
    # Add the outlook CDI variable to the dataset
    ds_outlook['outlook_cdi'] = xr.DataArray(
        data=outlook_cdi,
        dims=['latitude', 'longitude', 'time'],
        attrs={
            'long_name': 'Drought outlook CDI',
            'units': '%',
            'valid_min': 0.0,
            'valid_max': 100.0,
            'description': 'Historical percentage consistency of drought status between month pairs'
        }
    )
    
    # Add the drought percentage variable to the dataset
    ds_outlook['drought_pct'] = xr.DataArray(
        data=drought_pct_values,
        dims=['latitude', 'longitude', 'time'],
        attrs={
            'long_name': 'Drought percentage',
            'units': '%',
            'valid_min': 0.0,
            'valid_max': 100.0,
            'description': 'Historical percentage of drought occurrence (CDI ≤ threshold)'
        }
    )
    
    return ds_outlook, stats

def save_outlook_cdi(ds_outlook, output_file):
    """
    Save the outlook CDI dataset to a NetCDF file.
    
    Args:
        ds_outlook (xarray.Dataset): Dataset with outlook CDI values
        output_file (str): Path to save the output file
        
    Returns:
        str: Path to the saved file
    """
    logger.info(f"Step 3: Saving outlook CDI to NetCDF file: {output_file}")
    write_history(f"\nSTEP 3: Saving outlook CDI to NetCDF file: {output_file}")
    
    # Make sure the directory exists if needed
    directory = os.path.dirname(output_file)
    if directory:
        os.makedirs(directory, exist_ok=True)
    
    # Save the dataset
    ds_outlook.to_netcdf(output_file)
    
    # Get file size
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    
    logger.info(f"  File saved successfully: {output_file} ({file_size_mb:.1f} MB)")
    logger.info(f"  Variables: {list(ds_outlook.data_vars)}")
    logger.info(f"  Time steps: {ds_outlook.dims['time']}")
    write_history(f"File saved successfully: {output_file} ({file_size_mb:.1f} MB)")
    write_history(f"Variables: {list(ds_outlook.data_vars)}")
    write_history(f"Time steps: {ds_outlook.dims['time']}")
    
    return output_file

def create_color_maps():
    """
    Create custom colormaps for different visualization types.
    
    Returns:
        dict: Dictionary of custom colormaps and norms
    """
    # Create colormap for consistency percentage
    consistency_colors = [
        '#FFFFFF',  # White (0%)
        '#FCDE66',  # Light yellow (~20%)
        '#FDB249',  # Light orange (~40%)
        '#F77C1C',  # Orange (~60%)
        '#E04E15',  # Dark orange (~80%)
        '#B01115',  # Red (~100%)
    ]
    consistency_cmap = mcolors.ListedColormap(consistency_colors)
    consistency_bounds = [0, 20, 40, 60, 80, 100]
    consistency_norm = mcolors.BoundaryNorm(consistency_bounds, consistency_cmap.N)
    
    # Create colormap for drought percentage
    drought_colors = [
        '#FFFFFF',  # White (0%)
        '#EDF8FB',  # Light blue (~20%)
        '#BFDCEB',  # Blue (~40%)
        '#67A9CF',  # Medium blue (~60%)
        '#2166AC',  # Dark blue (~80%)
        '#053061',  # Navy (~100%)
    ]
    drought_cmap = mcolors.ListedColormap(drought_colors)
    drought_bounds = [0, 20, 40, 60, 80, 100]
    drought_norm = mcolors.BoundaryNorm(drought_bounds, drought_cmap.N)
    
    # Create colormap for binary drought status
    binary_colors = ['#B0E2FF', '#E25822']  # Light blue for no drought, orange for drought
    binary_cmap = mcolors.ListedColormap(binary_colors)
    binary_bounds = [0, 0.5, 1]
    binary_norm = mcolors.BoundaryNorm(binary_bounds, binary_cmap.N)
    
    return {
        'consistency': {
            'cmap': consistency_cmap,
            'norm': consistency_norm,
            'bounds': consistency_bounds,
            'label': 'Drought Consistency Percentage (%)'
        },
        'drought_pct': {
            'cmap': drought_cmap,
            'norm': drought_norm,
            'bounds': drought_bounds,
            'label': 'Drought Percentage (%)'
        },
        'binary': {
            'cmap': binary_cmap,
            'norm': binary_norm,
            'bounds': binary_bounds,
            'label': 'Drought Status'
        }
    }

def generate_maps_parallel(stats, output_dir="maps"):
    """
    Generate maps for each month pair showing different metrics (consistency and drought percentage).
    
    Args:
        stats (dict): Statistics dictionary with percentage data
        output_dir (str): Directory to save the maps
        
    Returns:
        dict: Dictionary of maps organized by type and month pair
    """
    logger.info(f"Step 4: Generating maps for each month pair")
    write_history(f"\nSTEP 4: Generating maps for each month pair in {output_dir}")
    
    # Make sure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "consistency"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "drought_percentage"), exist_ok=True)
    
    # Dictionary to store map file paths
    map_files = {
        'consistency': {},
        'drought_pct': {},
        'combined': []
    }
    
    # Create custom colormaps
    colormaps = create_color_maps()
    
    # Process each month pair
    for pair_name, pair_stats in stats['month_pair_stats'].items():
        # Skip if we don't have percentage data
        if 'percentage_data' not in pair_stats:
            logger.warning(f"  No percentage data for {pair_name}, skipping map generation")
            continue
        
        logger.info(f"  Generating maps for {pair_name}")
        write_history(f"Generating maps for {pair_name}")
        
        # Generate consistency percentage map
        consistency_map = generate_single_map(
            pair_name, 
            pair_stats, 
            os.path.join(output_dir, "consistency"),
            "consistency", 
            pair_stats['percentage_data'],
            colormaps['consistency'],
            f"Drought Outlook: {pair_name}\nHistorical Consistency ({pair_stats['num_years']} years)"
        )
        map_files['consistency'][pair_name] = consistency_map
        
        # Generate drought percentage map
        drought_map = generate_single_map(
            pair_name, 
            pair_stats, 
            os.path.join(output_dir, "drought_percentage"),
            "drought_pct",
            pair_stats['drought_pct_data'],
            colormaps['drought_pct'],
            f"Drought Outlook: {pair_name}\nHistorical Drought Percentage ({pair_stats['num_years']} years)"
        )
        map_files['drought_pct'][pair_name] = drought_map
    
    # Generate combined maps for each type
    for map_type in ['consistency', 'drought_pct']:
        combined_file = generate_combined_map(
            stats, 
            os.path.join(output_dir, f"all_month_pairs_{map_type}.png"),
            map_type,
            colormaps[map_type]
        )
        map_files['combined'].append(combined_file)
    
    return map_files

def generate_single_map(pair_name, pair_stats, output_dir, map_type, data, colormap_info, title):
    """
    Generate a single map for a month pair and specific data type.
    
    Args:
        pair_name (str): Name of the month pair
        pair_stats (dict): Statistics for the month pair
        output_dir (str): Directory to save the map
        map_type (str): Type of map ('consistency' or 'drought_pct')
        data (numpy.ndarray): Data to plot
        colormap_info (dict): Colormap information
        title (str): Title for the map
        
    Returns:
        str: Path to the generated map
    """
    # Create a figure with a map projection
    fig = plt.figure(figsize=(12, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # Set the extent to focus on Australia
    ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
    
    # Add coastlines and borders
    ax.coastlines(resolution='50m', color='black', linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
    
    # Add gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                     linewidth=0.5, color='gray', linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    # Plot the data
    im = ax.pcolormesh(
        pair_stats['longitude'], pair_stats['latitude'], data,
        cmap=colormap_info['cmap'], norm=colormap_info['norm'], transform=ccrs.PlateCarree()
    )
    
    # Add a colorbar with custom ticks and labels
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8, 
                       ticks=[10, 30, 50, 70, 90])
    cbar.set_label(colormap_info['label'])
    
    # Add a title
    plt.title(title)
    
    # Add text with statistics
    if map_type == 'consistency':
        text = (
            f"Min: {pair_stats['min_percent']:.1f}%\n"
            f"Max: {pair_stats['max_percent']:.1f}%\n"
            f"Mean: {pair_stats['mean_percent']:.1f}%"
        )
    else:  # drought_pct
        text = (
            f"Min: {pair_stats['min_drought_pct']:.1f}%\n"
            f"Max: {pair_stats['max_drought_pct']:.1f}%\n"
            f"Mean: {pair_stats['mean_drought_pct']:.1f}%"
        )
    
    plt.figtext(0.15, 0.02, text, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    
    # Create the filename
    month1, month2 = pair_name.split('-')
    filename = f"{map_type}_{month1.lower()}_{month2.lower()}.png"
    file_path = os.path.join(output_dir, filename)
    
    # Save the figure
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    return file_path

def generate_combined_map(stats, output_file, map_type, colormap_info):
    """
    Generate a combined map showing all month pairs for a specific data type.
    
    Args:
        stats (dict): Statistics dictionary
        output_file (str): Path to save the combined map
        map_type (str): Type of map ('consistency' or 'drought_pct')
        colormap_info (dict): Colormap information
        
    Returns:
        str: Path to the generated map
    """
    # Create a figure with subplots
    fig = plt.figure(figsize=(20, 16))
    
    # Sort month pairs in calendar order
    month_pairs = []
    for pair_name in stats['month_pair_stats']:
        month1, month2 = pair_name.split('-')
        month1_idx = list(calendar.month_name).index(month1)
        month_pairs.append((month1_idx, pair_name))
    
    month_pairs.sort()
    
    # Process each month pair
    for i, (_, pair_name) in enumerate(month_pairs):
        pair_stats = stats['month_pair_stats'][pair_name]
        
        # Skip if we don't have the required data
        if map_type == 'consistency' and 'percentage_data' not in pair_stats:
            continue
        if map_type == 'drought_pct' and 'drought_pct_data' not in pair_stats:
            continue
        
        # Get the data to plot
        data = pair_stats['percentage_data'] if map_type == 'consistency' else pair_stats['drought_pct_data']
        
        # Create a subplot
        ax = fig.add_subplot(3, 4, i+1, projection=ccrs.PlateCarree())
        
        # Set the extent to focus on Australia
        ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
        
        # Add coastlines
        ax.coastlines(resolution='50m', color='black', linewidth=0.5)
        
        # Plot the data
        im = ax.pcolormesh(
            pair_stats['longitude'], pair_stats['latitude'], data,
            cmap=colormap_info['cmap'], norm=colormap_info['norm'], transform=ccrs.PlateCarree()
        )
        
        # Add a title
        ax.set_title(f"{pair_name}")
        
        # Add gridlines (simplified for the small maps)
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False,
                         linewidth=0.3, color='gray', linestyle=':')
    
    # Add a colorbar at the bottom
    cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.02])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', 
                       ticks=[10, 30, 50, 70, 90])
    cbar.set_label(colormap_info['label'])
    
    # Add an overall title
    if map_type == 'consistency':
        plt.suptitle('Drought Outlook: Historical Consistency by Month Pair', fontsize=16, y=0.98)
    else:  # drought_pct
        plt.suptitle('Drought Outlook: Historical Drought Percentage by Month Pair', fontsize=16, y=0.98)
    
    # Save the figure
    plt.savefig(output_file, dpi=200, bbox_inches='tight')
    plt.close(fig)
    
    return output_file

def generate_monthly_maps_parallel(ds_outlook, output_dir="monthly_maps", max_maps=None, num_processes=None):
    """
    Generate maps for each month in the outlook dataset using parallel processing.
    
    Args:
        ds_outlook (xarray.Dataset): Dataset with outlook CDI values
        output_dir (str): Directory to save the maps
        max_maps (int, optional): Maximum number of maps to generate (None = all)
        num_processes (int, optional): Number of processes to use (None = all available)
        
    Returns:
        dict: Dictionary with maps organized by type
    """
    logger.info(f"Step 5: Generating monthly maps in parallel")
    write_history(f"\nSTEP 5: Generating monthly maps in parallel")
    
    # Make sure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "consistency"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "drought_percentage"), exist_ok=True)
    
    # Get coordinates and time values
    lat = ds_outlook.latitude.values
    lon = ds_outlook.longitude.values
    times = pd.to_datetime(ds_outlook.time.values)
    
    # Determine how many maps to generate
    if max_maps is None:
        times_to_process = times
    else:
        # If max_maps specified, select a sample across the time range
        if len(times) > max_maps:
            # Select evenly spaced times
            indices = np.linspace(0, len(times) - 1, max_maps, dtype=int)
            times_to_process = [times[i] for i in indices]
        else:
            times_to_process = times
    
    logger.info(f"  Generating {len(times_to_process)} monthly maps for both consistency and drought percentage")
    write_history(f"Generating {len(times_to_process)} monthly maps for both metrics")
    
    # Create custom colormaps
    colormaps = create_color_maps()
    
    # Prepare tasks for parallel processing
    map_tasks = []
    for i, time in enumerate(times_to_process):
        time_idx = np.where(ds_outlook.time.values == np.datetime64(time))[0][0]
        
        # Add task for consistency map
        map_tasks.append({
            'time': time,
            'time_idx': time_idx,
            'map_type': 'consistency',
            'data_var': 'outlook_cdi',
            'output_dir': os.path.join(output_dir, "consistency")
        })
        
        # Add task for drought percentage map
        map_tasks.append({
            'time': time,
            'time_idx': time_idx,
            'map_type': 'drought_pct',
            'data_var': 'drought_pct',
            'output_dir': os.path.join(output_dir, "drought_percentage")
        })
    
    # Set up multiprocessing
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    # Create a partial function with fixed arguments
    process_func = partial(
        generate_monthly_map,
        ds_outlook=ds_outlook,
        lat=lat,
        lon=lon,
        colormaps=colormaps
    )
    
    # Process maps in parallel
    logger.info(f"  Using {num_processes} processes for generating maps")
    write_history(f"Using {num_processes} processes for generating maps")
    
    start_time = time.time()
    
    # Using the Pool context manager
    with mp.Pool(processes=num_processes) as pool:
        map_files = list(tqdm(
            pool.imap(process_func, map_tasks),
            total=len(map_tasks),
            desc="Generating monthly maps"
        ))
    
    end_time = time.time()
    
    logger.info(f"  Map generation completed in {end_time - start_time:.2f} seconds")
    write_history(f"Map generation completed in {end_time - start_time:.2f} seconds")
    
    # Organize results
    results = {
        'consistency': [f for f in map_files if 'consistency' in f],
        'drought_pct': [f for f in map_files if 'drought_percentage' in f],
        'sample': []
    }
    
    # Create sample maps
    for map_type in ['consistency', 'drought_pct']:
        if len(results[map_type]) > 0:
            sample_file = generate_sample_map(
                ds_outlook, 
                times, 
                os.path.join(output_dir, f"sample_{map_type}.png"),
                map_type,
                colormaps[map_type]
            )
            results['sample'].append(sample_file)
    
    return results

def generate_monthly_map(task, ds_outlook, lat, lon, colormaps):
    """
    Generate a map for a single month and data type.
    
    Args:
        task (dict): Task dictionary with parameters
        ds_outlook (xarray.Dataset): Dataset with outlook CDI values
        lat (numpy.ndarray): Latitude values
        lon (numpy.ndarray): Longitude values
        colormaps (dict): Dictionary of colormaps
        
    Returns:
        str: Path to the generated map
    """
    time = task['time']
    time_idx = task['time_idx']
    map_type = task['map_type']
    data_var = task['data_var']
    output_dir = task['output_dir']
    
    # Get the data for this time
    data = ds_outlook[data_var].values[:, :, time_idx]
    
    # Format the date string
    date_str = time.strftime('%Y-%m')
    month_name = time.strftime('%B %Y')
    
    # Determine the next month (for title)
    next_month = (time.month % 12) + 1
    next_month_year = time.year if next_month > time.month else time.year + 1
    next_month_name = datetime(next_month_year, next_month, 1).strftime('%B %Y')
    
    # Create a figure
    fig = plt.figure(figsize=(12, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # Set the extent to focus on Australia
    ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
    
    # Add coastlines and borders
    ax.coastlines(resolution='50m', color='black', linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
    
    # Add gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                     linewidth=0.5, color='gray', linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    # Get colormap info
    colormap_info = colormaps[map_type]
    
    # Plot the data
    im = ax.pcolormesh(lon, lat, data, cmap=colormap_info['cmap'], norm=colormap_info['norm'], transform=ccrs.PlateCarree())
    
    # Add a colorbar
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8, 
                       ticks=[10, 30, 50, 70, 90])
    cbar.set_label(colormap_info['label'])
    
    # Add a title
    if map_type == 'consistency':
        title = f"Drought Outlook: {month_name} → {next_month_name}\nHistorical Consistency"
    else:  # drought_pct
        title = f"Drought Outlook: {month_name} → {next_month_name}\nHistorical Drought Percentage"
    
    plt.title(title, fontsize=14)
    
    # Add explanatory text
    if map_type == 'consistency':
        text = (
            f"This map shows the historical consistency of drought status\n"
            f"between {time.strftime('%B')} and {datetime(next_month_year, next_month, 1).strftime('%B')}\n"
            f"based on historical CDI data.\n\n"
            f"Higher values (darker colors) indicate areas where drought status\n"
            f"typically remains consistent between these months."
        )
    else:  # drought_pct
        text = (
            f"This map shows the historical percentage of drought occurrence\n"
            f"for {time.strftime('%B')} based on historical CDI data.\n\n"
            f"Higher values (darker colors) indicate areas with more frequent\n"
            f"drought conditions in this month historically."
        )
    
    plt.figtext(0.5, 0.02, text, fontsize=12, ha='center', bbox=dict(facecolor='white', alpha=0.8))
    
    # Create the filename
    filename = f"{map_type}_{date_str}.png"
    file_path = os.path.join(output_dir, filename)
    
    # Save the figure
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    return file_path

def generate_sample_map(ds_outlook, times, output_file, map_type, colormap_info):
    """
    Generate a multi-panel map showing a sample of months for a specific data type.
    
    Args:
        ds_outlook (xarray.Dataset): Dataset with outlook CDI values
        times (list): List of all times
        output_file (str): Path to save the output file
        map_type (str): Type of map ('consistency' or 'drought_pct')
        colormap_info (dict): Colormap information
        
    Returns:
        str: Path to the generated map
    """
    # Get coordinates
    lat = ds_outlook.latitude.values
    lon = ds_outlook.longitude.values
    
    # Select a sample of times (12 months across the time range)
    if len(times) > 12:
        sample_indices = np.linspace(0, len(times) - 1, 12, dtype=int)
        sample_times = [times[i] for i in sample_indices]
    else:
        sample_times = times
    
    # Get the data variable name
    data_var = 'outlook_cdi' if map_type == 'consistency' else 'drought_pct'
    
    # Create a figure with subplots
    fig = plt.figure(figsize=(20, 16))
    
    # Process each sample time
    for i, time in enumerate(sample_times):
        # Get the data for this time
        time_idx = np.where(ds_outlook.time.values == np.datetime64(time))[0][0]
        data = ds_outlook[data_var].values[:, :, time_idx]
        
        # Format the date string
        month_name = time.strftime('%b %Y')
        
        # Create a subplot
        ax = fig.add_subplot(3, 4, i+1, projection=ccrs.PlateCarree())
        
        # Set the extent to focus on Australia
        ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
        
        # Add coastlines
        ax.coastlines(resolution='50m', color='black', linewidth=0.5)
        
        # Plot the data
        im = ax.pcolormesh(lon, lat, data, cmap=colormap_info['cmap'], norm=colormap_info['norm'], transform=ccrs.PlateCarree())
        
        # Add a title
        ax.set_title(f"{month_name}")
        
        # Add gridlines (simplified for the small maps)
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False,
                         linewidth=0.3, color='gray', linestyle=':')
    
    # Add a colorbar at the bottom
    cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.02])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', 
                       ticks=[10, 30, 50, 70, 90])
    cbar.set_label(colormap_info['label'])
    
    # Add an overall title
    if map_type == 'consistency':
        plt.suptitle('Drought Outlook: Sample Months - Consistency Percentage', fontsize=16, y=0.98)
    else:  # drought_pct
        plt.suptitle('Drought Outlook: Sample Months - Drought Percentage', fontsize=16, y=0.98)
    
    # Save the figure
    plt.savefig(output_file, dpi=200, bbox_inches='tight')
    plt.close(fig)
    
    return output_file

def generate_yearly_maps_parallel(ds, output_base_dir="yearly_maps", drought_threshold=0.3, num_processes=None):
    """
    Generate maps organized by year in parallel, each in its own folder.
    
    Args:
        ds (xarray.Dataset): Dataset with CDI data
        output_base_dir (str): Base directory for yearly maps
        drought_threshold (float): Threshold for drought classification
        num_processes (int, optional): Number of processes to use (None = all available)
        
    Returns:
        dict: Dictionary with information about generated maps
    """
    logger.info(f"Step 6: Generating yearly map collections in parallel")
    write_history(f"\nSTEP 6: Generating yearly map collections in parallel")
    
    # Create base directory
    os.makedirs(output_base_dir, exist_ok=True)
    
    # Extract dimensions and coordinates
    times = ds.time.values
    times_dt = pd.to_datetime(times)
    
    # Group times by year
    years_in_data = sorted(set(t.year for t in times_dt))
    write_history(f"Found data for {len(years_in_data)} years: {years_in_data}")
    
    # Create tasks for parallel processing
    year_tasks = []
    for year in years_in_data:
        year_dir = os.path.join(output_base_dir, f"{year}")
        os.makedirs(year_dir, exist_ok=True)
        
        # Filter times for this year
        year_times = [t for t in times_dt if t.year == year]
        year_indices = [i for i, t in enumerate(times_dt) if t.year == year]
        
        year_tasks.append({
            'year': year,
            'year_dir': year_dir,
            'year_times': year_times,
            'year_indices': year_indices
        })
    
    # Set up multiprocessing
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    # Create a partial function with fixed arguments
    process_func = partial(
        process_year_maps,
        ds=ds,
        drought_threshold=drought_threshold
    )
    
    # Process years in parallel
    logger.info(f"  Using {num_processes} processes for generating yearly maps")
    write_history(f"Using {num_processes} processes for generating yearly maps")
    
    start_time = time.time()
    
    # Using the Pool context manager
    with mp.Pool(processes=num_processes) as pool:
        year_results = list(tqdm(
            pool.imap(process_func, year_tasks),
            total=len(year_tasks),
            desc="Generating yearly maps"
        ))
    
    end_time = time.time()
    
    logger.info(f"  Yearly map generation completed in {end_time - start_time:.2f} seconds")
    write_history(f"Yearly map generation completed in {end_time - start_time:.2f} seconds")
    
    # Combine results
    results = {
        'total_maps': sum(len(maps) for year, maps in year_results),
        'years_processed': years_in_data,
        'map_paths': {year: maps for year, maps in year_results}
    }
    
    return results

def process_year_maps(task, ds, drought_threshold=0.3):
    """
    Process maps for a single year.
    
    Args:
        task (dict): Task dictionary with parameters
        ds (xarray.Dataset): Dataset with CDI data
        drought_threshold (float): Threshold for drought classification
        
    Returns:
        tuple: (year, year_maps) - Year and list of maps for that year
    """
    year = task['year']
    year_dir = task['year_dir']
    year_times = task['year_times']
    year_indices = task['year_indices']
    
    # Get coordinates
    latitude = ds.latitude.values
    longitude = ds.longitude.values
    
    # Create colormaps
    colormaps = create_color_maps()
    
    # List to store map info
    year_maps = []
    
    # Process each month in this year
    for month_idx, month_time in zip(year_indices, year_times):
        month_name = month_time.strftime('%B')
        
        # Get the CDI data for this month
        cdi_data = ds.cdi.values[:, :, month_idx]
        
        # Create a mask for valid data (between 0 and 1, not NaN)
        valid_mask = ~np.isnan(cdi_data) & (cdi_data >= 0) & (cdi_data <= 1)
        
        # Determine drought status (1 for drought, 0 for no drought)
        drought_status = np.full(cdi_data.shape, np.nan, dtype=np.float32)
        drought_status[valid_mask] = (cdi_data[valid_mask] <= drought_threshold).astype(np.float32)
        
        # Calculate drought percentage
        drought_count = np.sum(drought_status == 1)
        no_drought_count = np.sum(drought_status == 0)
        valid_count = np.sum(valid_mask)
        
        # Generate the map
        file_path = generate_yearly_month_map(
            year, month_time, longitude, latitude, drought_status, drought_count, no_drought_count,
            valid_count, colormaps['binary'], drought_threshold, year_dir
        )
        
        # Add to the list of maps for this year
        year_maps.append({
            'path': file_path,
            'filename': os.path.basename(file_path),
            'month': month_name,
            'year': year,
            'month_num': month_time.month,
            'drought_count': int(drought_count),
            'no_drought_count': int(no_drought_count),
            'valid_count': int(valid_count),
            'drought_pct': (drought_count / valid_count * 100) if valid_count > 0 else 0
        })
    
    # Create a combined map for this year
    if year_maps:
        combined_file = generate_yearly_combined_map(year, year_maps, year_dir)
        
        # Add the combined map to the year maps
        year_maps.append({
            'path': combined_file,
            'filename': os.path.basename(combined_file),
            'month': 'All',
            'year': year,
            'month_num': 0,
            'is_combined': True
        })
    
    return year, year_maps

def generate_yearly_month_map(year, month_time, longitude, latitude, drought_status, 
                             drought_count, no_drought_count, valid_count, 
                             colormap_info, drought_threshold, output_dir):
    """
    Generate a map for a specific month in a year.
    
    Args:
        year (int): Year
        month_time (datetime): Month datetime
        longitude (numpy.ndarray): Longitude values
        latitude (numpy.ndarray): Latitude values
        drought_status (numpy.ndarray): Drought status data
        drought_count (int): Count of drought cells
        no_drought_count (int): Count of no-drought cells
        valid_count (int): Count of valid cells
        colormap_info (dict): Colormap information
        drought_threshold (float): Threshold for drought classification
        output_dir (str): Directory to save the map
        
    Returns:
        str: Path to the generated map
    """
    month_name = month_time.strftime('%B')
    
    # Create a figure
    fig = plt.figure(figsize=(12, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # Set the extent to focus on Australia
    ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
    
    # Add coastlines and borders
    ax.coastlines(resolution='50m', color='black', linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
    
    # Add gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                     linewidth=0.5, color='gray', linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    
    # Plot the data
    im = ax.pcolormesh(longitude, latitude, drought_status, cmap=colormap_info['cmap'], 
                      norm=colormap_info['norm'], transform=ccrs.PlateCarree())
    
    # Add a colorbar
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8, ticks=[0.25, 0.75])
    cbar.set_ticklabels(['No Drought', 'Drought'])
    cbar.set_label(f'Drought Status (Threshold: CDI ≤ {drought_threshold})')
    
    # Add a title
    plt.title(f"Drought Status: {month_name} {year}", fontsize=14)
    
    # Add statistics
    stats_text = (
        f"Valid cells: {valid_count}\n"
        f"Drought cells: {drought_count} ({drought_count/valid_count*100:.1f}%)\n"
        f"No-drought cells: {no_drought_count} ({no_drought_count/valid_count*100:.1f}%)"
    )
    plt.figtext(0.15, 0.02, stats_text, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    
    # Create the filename
    filename = f"drought_status_{year}_{month_time.strftime('%m')}_{month_name}.png"
    file_path = os.path.join(output_dir, filename)
    
    # Save the figure
    plt.savefig(file_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    return file_path

def generate_yearly_combined_map(year, year_maps, output_dir):
    """
    Generate a combined map for a year.
    
    Args:
        year (int): Year
        year_maps (list): List of maps for the year
        output_dir (str): Directory to save the combined map
        
    Returns:
        str: Path to the combined map
    """
    # Sort maps by month
    month_maps = [m for m in year_maps if not m.get('is_combined', False)]
    month_maps.sort(key=lambda x: x['month_num'])
    
    # Determine layout
    num_maps = len(month_maps)
    grid_size = int(np.ceil(np.sqrt(num_maps)))
    rows = grid_size
    cols = grid_size
    
    # Create a figure with subplots
    fig = plt.figure(figsize=(cols*4, rows*3.5))
    
    # Process each month
    for i, map_info in enumerate(month_maps):
        if i < rows * cols:
            # Calculate row and column
            row = i // cols
            col = i % cols
            
            # Create a subplot
            ax = fig.add_subplot(rows, cols, i+1, projection=ccrs.PlateCarree())
            
            # Load the image
            img = plt.imread(map_info['path'])
            
            # Display the image
            ax.imshow(img, extent=[0, 1, 0, 1], transform=ax.transAxes)
            
            # Remove axis
            ax.axis('off')
            
            # Add a title
            ax.set_title(f"{map_info['month']} ({map_info['drought_pct']:.1f}% Drought)", fontsize=12)
    
    # Add an overall title
    plt.suptitle(f'Drought Status for {year}', fontsize=16, y=0.98)
    
    # Save the figure
    combined_file = os.path.join(output_dir, f"combined_{year}.png")
    plt.savefig(combined_file, dpi=200, bbox_inches='tight')
    plt.close(fig)
    
    return combined_file

def create_interactive_dashboard(yearly_results, month_pair_maps, monthly_maps, output_file="drought_dashboard.html"):
    """
    Create an interactive HTML dashboard with Bootstrap.
    
    Args:
        yearly_results (dict): Results from the yearly maps generation
        month_pair_maps (dict): Month pair maps by type
        monthly_maps (dict): Monthly maps by type
        output_file (str): Path to save the HTML report
        
    Returns:
        str: Path to the HTML dashboard
    """
    logger.info(f"Step 7: Creating interactive dashboard: {output_file}")
    write_history(f"\nSTEP 7: Creating interactive dashboard: {output_file}")
    
    # Create HTML content
    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Interactive Drought Analysis Dashboard</title>
        <!-- Bootstrap CSS -->
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha1/dist/css/bootstrap.min.css" rel="stylesheet">
        <!-- Font Awesome Icons -->
        <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css">
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                background-color: #f8f9fa;
                color: #343a40;
            }}
            .dashboard-header {{
                background-color: #343a40;
                color: white;
                padding: 20px 0;
                margin-bottom: 30px;
                box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            }}
            .nav-tabs {{
                border-bottom: 2px solid #dee2e6;
                margin-bottom: 20px;
            }}
            .nav-tabs .nav-link {{
                color: #495057;
                background-color: #f8f9fa;
                border: 1px solid transparent;
                border-top-left-radius: 0.25rem;
                border-top-right-radius: 0.25rem;
                padding: 12px 20px;
                font-weight: 500;
                transition: all 0.3s ease;
            }}
            .nav-tabs .nav-link:hover {{
                border-color: #e9ecef #e9ecef #dee2e6;
                background-color: #e9ecef;
            }}
            .nav-tabs .nav-link.active {{
                color: #495057;
                background-color: #fff;
                border-color: #dee2e6 #dee2e6 #fff;
                border-bottom: 3px solid #007bff;
            }}
            .tab-content {{
                background-color: white;
                border-radius: 0 0 5px 5px;
                padding: 25px;
                box-shadow: 0 1px 3px rgba(0, 0, 0, 0.1);
            }}
            .map-container {{
                margin-bottom: 30px;
                border-radius: 5px;
                overflow: hidden;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.1);
                background-color: white;
            }}
            .map-title {{
                background-color: #f8f9fa;
                padding: 15px;
                border-bottom: 1px solid #dee2e6;
                font-weight: 600;
            }}
            .map-image img {{
                width: 100%;
                display: block;
            }}
            .map-footer {{
                padding: 15px;
                font-size: 14px;
                border-top: 1px solid #dee2e6;
                background-color: #f8f9fa;
            }}
            .year-section {{
                margin-bottom: 40px;
            }}
            .btn-toggle {{
                margin-bottom: 20px;
            }}
            .btn-toggle .btn {{
                border-radius: 30px;
                padding: 8px 18px;
                margin-right: 5px;
                font-weight: 500;
            }}
            .map-controls {{
                margin-bottom: 20px;
                padding: 15px;
                background-color: #f8f9fa;
                border-radius: 5px;
                box-shadow: 0 1px 3px rgba(0, 0, 0, 0.1);
            }}
            .accordion-button:not(.collapsed) {{
                background-color: #e7f5ff;
                color: #0d6efd;
            }}
            .stat-card {{
                background-color: white;
                border-radius: 5px;
                padding: 20px;
                margin-bottom: 20px;
                box-shadow: 0 2px 4px rgba(0, 0, 0, 0.1);
                height: 100%;
            }}
            .stat-card .stat-value {{
                font-size: 2rem;
                font-weight: 700;
                margin-bottom: 10px;
                color: #0d6efd;
            }}
            .stat-card .stat-title {{
                font-size: 1rem;
                color: #6c757d;
                margin-bottom: 0;
            }}
            .stat-card .stat-icon {{
                font-size: 3rem;
                margin-bottom: 15px;
                color: #0d6efd;
                opacity: 0.7;
            }}
            footer {{
                background-color: #343a40;
                color: white;
                padding: 20px 0;
                margin-top: 50px;
            }}
            #yearSelector, #mapTypeSelector {{
                border-radius: 5px;
                padding: 10px;
                font-weight: 500;
            }}
            .nav-pills .nav-link {{
                margin-right: 5px;
                border-radius: 5px;
                padding: 8px 16px;
            }}
            .nav-pills .nav-link.active {{
                background-color: #007bff;
            }}
            .dropdown-menu {{
                max-height: 300px;
                overflow-y: auto;
            }}
        </style>
    </head>
    <body>
        <div class="dashboard-header">
            <div class="container">
                <div class="row align-items-center">
                    <div class="col-md-8">
                        <h1><i class="fas fa-cloud-rain me-3"></i>Drought Analysis Dashboard</h1>
                        <p class="lead mb-0">Interactive visualization of drought patterns across Australia</p>
                    </div>
                    <div class="col-md-4 text-end">
                        <button class="btn btn-light" type="button" data-bs-toggle="modal" data-bs-target="#infoModal">
                            <i class="fas fa-info-circle me-2"></i>About This Dashboard
                        </button>
                    </div>
                </div>
            </div>
        </div>

        <div class="container mb-5">
            <!-- Stats Row -->
            <div class="row mb-4">
                <div class="col-md-3">
                    <div class="stat-card text-center">
                        <div class="stat-icon">
                            <i class="fas fa-map"></i>
                        </div>
                        <div class="stat-value">{yearly_results['total_maps']}</div>
                        <p class="stat-title">Total Maps Generated</p>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card text-center">
                        <div class="stat-icon">
                            <i class="fas fa-calendar-alt"></i>
                        </div>
                        <div class="stat-value">{len(yearly_results['years_processed'])}</div>
                        <p class="stat-title">Years Analyzed</p>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card text-center">
                        <div class="stat-icon">
                            <i class="fas fa-tint"></i>
                        </div>
                        <div class="stat-value">0.3</div>
                        <p class="stat-title">Drought Threshold (CDI)</p>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card text-center">
                        <div class="stat-icon">
                            <i class="fas fa-chart-area"></i>
                        </div>
                        <div class="stat-value">Australia</div>
                        <p class="stat-title">Analysis Region</p>
                    </div>
                </div>
            </div>

            <!-- Main Tabs -->
            <ul class="nav nav-tabs" id="droughtTabs" role="tablist">
                <li class="nav-item" role="presentation">
                    <button class="nav-link active" id="monthly-tab" data-bs-toggle="tab" data-bs-target="#monthly" type="button" role="tab" aria-controls="monthly" aria-selected="true">
                        <i class="fas fa-calendar-day me-2"></i>Monthly Analysis
                    </button>
                </li>
                <li class="nav-item" role="presentation">
                    <button class="nav-link" id="yearly-tab" data-bs-toggle="tab" data-bs-target="#yearly" type="button" role="tab" aria-controls="yearly" aria-selected="false">
                        <i class="fas fa-calendar-alt me-2"></i>Yearly Analysis
                    </button>
                </li>
                <li class="nav-item" role="presentation">
                    <button class="nav-link" id="patterns-tab" data-bs-toggle="tab" data-bs-target="#patterns" type="button" role="tab" aria-controls="patterns" aria-selected="false">
                        <i class="fas fa-chart-line me-2"></i>Drought Patterns
                    </button>
                </li>
            </ul>

            <div class="tab-content" id="droughtTabsContent">
                <!-- Monthly Analysis Tab -->
                <div class="tab-pane fade show active" id="monthly" role="tabpanel" aria-labelledby="monthly-tab">
                    <div class="row mb-4">
                        <div class="col-12">
                            <div class="map-controls">
                                <div class="row align-items-center">
                                    <div class="col-md-6">
                                        <h5 class="mb-3">View Controls</h5>
                                        <div class="btn-group" role="group" aria-label="View toggle">
                                            <input type="radio" class="btn-check" name="viewType" id="consistencyView" autocomplete="off" checked>
                                            <label class="btn btn-outline-primary" for="consistencyView">
                                                <i class="fas fa-exchange-alt me-2"></i>Consistency View
                                            </label>
                                            <input type="radio" class="btn-check" name="viewType" id="droughtView" autocomplete="off">
                                            <label class="btn btn-outline-primary" for="droughtView">
                                                <i class="fas fa-tint me-2"></i>Drought Percentage View
                                            </label>
                                        </div>
                                    </div>
                                    <div class="col-md-6">
                                        <div class="form-group">
                                            <label for="monthPairSelector" class="form-label">Select Month Pair:</label>
                                            <select class="form-select" id="monthPairSelector">
                                                <option value="all">All Month Pairs (Overview)</option>
                                                <option value="Jan-Feb">January-February</option>
                                                <option value="Feb-Mar">February-March</option>
                                                <option value="Mar-Apr">March-April</option>
                                                <option value="Apr-May">April-May</option>
                                                <option value="May-Jun">May-June</option>
                                                <option value="Jun-Jul">June-July</option>
                                                <option value="Jul-Aug">July-August</option>
                                                <option value="Aug-Sep">August-September</option>
                                                <option value="Sep-Oct">September-October</option>
                                                <option value="Oct-Nov">October-November</option>
                                                <option value="Nov-Dec">November-December</option>
                                                <option value="Dec-Jan">December-January</option>
                                            </select>
                                        </div>
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>

                    <div class="row">
                        <div class="col-12">
                            <div class="map-container consistency-view">
                                <div class="map-title">
                                    <h4>Drought Consistency Pattern - All Month Pairs</h4>
                                </div>
                                <div class="map-image text-center">
                                    <img src="maps/all_month_pairs_consistency.png" class="img-fluid" alt="All Month Pairs - Consistency">
                                </div>
                                <div class="map-footer">
                                    <div class="row">
                                        <div class="col-md-8">
                                            <p>This map shows the historical consistency of drought status between consecutive months across Australia. Higher values (darker colors) indicate areas where drought status typically remains consistent from one month to the next.</p>
                                        </div>
                                        <div class="col-md-4 text-end">
                                            <a href="maps/all_month_pairs_consistency.png" class="btn btn-sm btn-outline-primary" target="_blank">
                                                <i class="fas fa-search-plus me-1"></i>View Full Size
                                            </a>
                                        </div>
                                    </div>
                                </div>
                            </div>

                            <div class="map-container drought-view" style="display: none;">
                                <div class="map-title">
                                    <h4>Drought Percentage Pattern - All Month Pairs</h4>
                                </div>
                                <div class="map-image text-center">
                                    <img src="maps/all_month_pairs_drought_pct.png" class="img-fluid" alt="All Month Pairs - Drought Percentage">
                                </div>
                                <div class="map-footer">
                                    <div class="row">
                                        <div class="col-md-8">
                                            <p>This map shows the historical percentage of drought occurrence for each month across Australia. Higher values (darker colors) indicate areas with more frequent drought conditions historically.</p>
                                        </div>
                                        <div class="col-md-4 text-end">
                                            <a href="maps/all_month_pairs_drought_pct.png" class="btn btn-sm btn-outline-primary" target="_blank">
                                                <i class="fas fa-search-plus me-1"></i>View Full Size
                                            </a>
                                        </div>
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>

                    <div class="row mt-4">
                        <div class="col-12">
                            <h4 class="mb-3">Monthly Sample Maps</h4>
                            <div class="map-container consistency-view">
                                <div class="map-image text-center">
                                    <img src="monthly_maps/sample_consistency.png" class="img-fluid" alt="Monthly Sample - Consistency">
                                </div>
                            </div>
                            <div class="map-container drought-view" style="display: none;">
                                <div class="map-image text-center">
                                    <img src="monthly_maps/sample_drought_pct.png" class="img-fluid" alt="Monthly Sample - Drought Percentage">
                                </div>
                            </div>
                        </div>
                    </div>
                </div>

                <!-- Yearly Analysis Tab -->
                <div class="tab-pane fade" id="yearly" role="tabpanel" aria-labelledby="yearly-tab">
                    <div class="row mb-4">
                        <div class="col-12">
                            <div class="map-controls">
                                <div class="row align-items-center">
                                    <div class="col-md-6">
                                        <h5 class="mb-3">Year Selection</h5>
                                        <div class="form-group">
                                            <select class="form-select" id="yearSelector">
                                                <option value="">Select a Year</option>
    """
    
    # Add year options
    for year in sorted(yearly_results['years_processed']):
        html += f'                                                <option value="{year}">{year}</option>\n'
    
    html += """
                                            </select>
                                        </div>
                                    </div>
                                    <div class="col-md-6">
                                        <div class="btn-group" role="group" aria-label="Map type toggle">
                                            <button type="button" class="btn btn-outline-primary active" id="combinedMapBtn">
                                                <i class="fas fa-th me-2"></i>Combined View
                                            </button>
                                            <button type="button" class="btn btn-outline-primary" id="individualMapBtn">
                                                <i class="fas fa-calendar-day me-2"></i>Individual Months
                                            </button>
                                        </div>
                                    </div>
                                </div>
                            </div>
                        </div>
                    </div>

                    <div class="row">
                        <div class="col-12">
                            <div id="yearlyMapContainer" class="map-container text-center">
                                <div class="map-title">
                                    <h4>Please select a year from the dropdown above</h4>
                                </div>
                                <div class="map-image text-center" id="yearlyMapImage">
                                    <!-- Map will be loaded here dynamically -->
                                </div>
                            </div>
                        </div>
                    </div>

                    <div class="row mt-4" id="individualMonthsContainer" style="display: none;">
                        <div class="col-12">
                            <h4 class="mb-3">Individual Months</h4>
                            <div class="row" id="monthlyMapsRow">
                                <!-- Individual month maps will be loaded here dynamically -->
                            </div>
                        </div>
                    </div>
                </div>

                <!-- Drought Patterns Tab -->
                <div class="tab-pane fade" id="patterns" role="tabpanel" aria-labelledby="patterns-tab">
                    <div class="row mb-4">
                        <div class="col-12">
                            <div class="alert alert-info">
                                <i class="fas fa-info-circle me-2"></i>This tab shows patterns of drought occurrence and consistency across different seasons in Australia.
                            </div>
                        </div>
                    </div>

                    <div class="row mb-4">
                        <div class="col-md-6">
                            <div class="map-container">
                                <div class="map-title">
                                    <h4>Summer Pattern (Dec-Feb)</h4>
                                </div>
                                <div class="map-image text-center">
                                    <img src="maps/consistency/consistency_december_january.png" class="img-fluid" alt="Summer Pattern">
                                </div>
                                <div class="map-footer">
                                    <p>Drought consistency pattern during summer months.</p>
                                </div>
                            </div>
                        </div>
                        <div class="col-md-6">
                            <div class="map-container">
                                <div class="map-title">
                                    <h4>Winter Pattern (Jun-Aug)</h4>
                                </div>
                                <div class="map-image text-center">
                                    <img src="maps/consistency/consistency_june_july.png" class="img-fluid" alt="Winter Pattern">
                                </div>
                                <div class="map-footer">
                                    <p>Drought consistency pattern during winter months.</p>
                                </div>
                            </div>
                        </div>
                    </div>

                    <div class="row">
                        <div class="col-md-6">
                            <div class="map-container">
                                <div class="map-title">
                                    <h4>Spring Pattern (Sep-Nov)</h4>
                                </div>
                                <div class="map-image text-center">
                                    <img src="maps/consistency/consistency_september_october.png" class="img-fluid" alt="Spring Pattern">
                                </div>
                                <div class="map-footer">
                                    <p>Drought consistency pattern during spring months.</p>
                                </div>
                            </div>
                        </div>
                        <div class="col-md-6">
                            <div class="map-container">
                                <div class="map-title">
                                    <h4>Autumn Pattern (Mar-May)</h4>
                                </div>
                                <div class="map-image text-center">
                                    <img src="maps/consistency/consistency_march_april.png" class="img-fluid" alt="Autumn Pattern">
                                </div>
                                <div class="map-footer">
                                    <p>Drought consistency pattern during autumn months.</p>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>
            </div>
        </div>

        <!-- About Modal -->
        <div class="modal fade" id="infoModal" tabindex="-1" aria-labelledby="infoModalLabel" aria-hidden="true">
            <div class="modal-dialog modal-lg">
                <div class="modal-content">
                    <div class="modal-header bg-primary text-white">
                        <h5 class="modal-title" id="infoModalLabel">About This Dashboard</h5>
                        <button type="button" class="btn-close" data-bs-dismiss="modal" aria-label="Close"></button>
                    </div>
                    <div class="modal-body">
                        <h5>Drought Analysis Methodology</h5>
                        <p>This dashboard provides an interactive visualization of drought patterns across Australia based on the Climate Drought Index (CDI).</p>
                        
                        <h6>Key Concepts:</h6>
                        <ul>
                            <li><strong>Drought Classification:</strong> CDI values ≤ 0.3 indicate drought conditions (1), values > 0.3 indicate non-drought conditions (0).</li>
                            <li><strong>Consistency Analysis:</strong> Compares drought status between consecutive months. If both months show consistent status, the result = 1. If months show inconsistent status, the result = 0.</li>
                            <li><strong>Drought Percentage:</strong> Shows how often a location experiences drought conditions historically.</li>
                        </ul>
                        
                        <h6>Data Sources:</h6>
                        <p>This analysis uses historical CDI data from 1998 to 2018, providing a 21-year climatological perspective on drought patterns.</p>
                        
                        <h6>View Types:</h6>
                        <ul>
                            <li><strong>Consistency View:</strong> Shows areas where drought status typically remains the same between consecutive months.</li>
                            <li><strong>Drought Percentage View:</strong> Shows areas that experience drought conditions more frequently.</li>
                            <li><strong>Yearly View:</strong> Shows the actual drought status for each month in a specific year.</li>
                        </ul>
                    </div>
                    <div class="modal-footer">
                        <button type="button" class="btn btn-secondary" data-bs-dismiss="modal">Close</button>
                    </div>
                </div>
            </div>
        </div>

        <!-- Footer -->
        <footer>
            <div class="container">
                <div class="row">
                    <div class="col-md-6">
                        <p class="mb-0">Drought Analysis Dashboard</p>
                        <p class="mb-0 small">Created: {datetime.now().strftime('%Y-%m-%d')}</p>
                    </div>
                    <div class="col-md-6 text-end">
                        <p class="mb-0 small">Based on CDI data from 1998-2018</p>
                    </div>
                </div>
            </div>
        </footer>

        <!-- JavaScript Bundle with Popper -->
        <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha1/dist/js/bootstrap.bundle.min.js"></script>
        
        <script>
            // Year data (will be replaced dynamically)
            const yearData = {
    """
    
    # Add year data for JavaScript
    for year, maps in yearly_results['map_paths'].items():
        html += f'                "{year}": {{\n'
        html += f'                    "combined": "{next((m["path"] for m in maps if m.get("is_combined", False)), "")}",\n'
        html += f'                    "months": [\n'
        
        # Add monthly maps for this year
        month_maps = [m for m in maps if not m.get('is_combined', False)]
        month_maps.sort(key=lambda x: x['month_num'])
        
        for month_map in month_maps:
            html += f'                        {{\n'
            html += f'                            "path": "{month_map["path"]}",\n'
            html += f'                            "month": "{month_map["month"]}",\n'
            html += f'                            "droughtPercent": {month_map.get("drought_pct", 0):.1f}\n'
            html += f'                        }},\n'
        
        html += f'                    ]\n'
        html += f'                }},\n'
    
    html += """
            };

            // Month pair data
            const monthPairData = {
                "consistency": {
                    "all": "maps/all_month_pairs_consistency.png",
                    "Jan-Feb": "maps/consistency/consistency_january_february.png",
                    "Feb-Mar": "maps/consistency/consistency_february_march.png",
                    "Mar-Apr": "maps/consistency/consistency_march_april.png",
                    "Apr-May": "maps/consistency/consistency_april_may.png",
                    "May-Jun": "maps/consistency/consistency_may_june.png",
                    "Jun-Jul": "maps/consistency/consistency_june_july.png",
                    "Jul-Aug": "maps/consistency/consistency_july_august.png",
                    "Aug-Sep": "maps/consistency/consistency_august_september.png",
                    "Sep-Oct": "maps/consistency/consistency_september_october.png",
                    "Oct-Nov": "maps/consistency/consistency_october_november.png",
                    "Nov-Dec": "maps/consistency/consistency_november_december.png",
                    "Dec-Jan": "maps/consistency/consistency_december_january.png"
                },
                "drought_pct": {
                    "all": "maps/all_month_pairs_drought_pct.png",
                    "Jan-Feb": "maps/drought_percentage/drought_pct_january_february.png",
                    "Feb-Mar": "maps/drought_percentage/drought_pct_february_march.png",
                    "Mar-Apr": "maps/drought_percentage/drought_pct_march_april.png",
                    "Apr-May": "maps/drought_percentage/drought_pct_april_may.png",
                    "May-Jun": "maps/drought_percentage/drought_pct_may_june.png",
                    "Jun-Jul": "maps/drought_percentage/drought_pct_june_july.png",
                    "Jul-Aug": "maps/drought_percentage/drought_pct_july_august.png",
                    "Aug-Sep": "maps/drought_percentage/drought_pct_august_september.png",
                    "Sep-Oct": "maps/drought_percentage/drought_pct_september_october.png",
                    "Oct-Nov": "maps/drought_percentage/drought_pct_october_november.png",
                    "Nov-Dec": "maps/drought_percentage/drought_pct_november_december.png",
                    "Dec-Jan": "maps/drought_percentage/drought_pct_december_january.png"
                }
            };

            document.addEventListener('DOMContentLoaded', function() {
                // View toggle handlers
                document.getElementById('consistencyView').addEventListener('change', function() {
                    document.querySelectorAll('.consistency-view').forEach(el => el.style.display = 'block');
                    document.querySelectorAll('.drought-view').forEach(el => el.style.display = 'none');
                    updateMonthPairMap();
                });

                document.getElementById('droughtView').addEventListener('change', function() {
                    document.querySelectorAll('.consistency-view').forEach(el => el.style.display = 'none');
                    document.querySelectorAll('.drought-view').forEach(el => el.style.display = 'block');
                    updateMonthPairMap();
                });

                // Month pair selector
                document.getElementById('monthPairSelector').addEventListener('change', function() {
                    updateMonthPairMap();
                });

                function updateMonthPairMap() {
                    const monthPair = document.getElementById('monthPairSelector').value;
                    const viewType = document.getElementById('consistencyView').checked ? 'consistency' : 'drought_pct';
                    
                    // Get the map containers
                    const consistencyContainer = document.querySelector('.consistency-view');
                    const droughtContainer = document.querySelector('.drought-view');
                    
                    // Update the titles
                    const titlePrefix = monthPair === 'all' ? 'All Month Pairs' : monthPair;
                    consistencyContainer.querySelector('.map-title h4').textContent = `Drought Consistency Pattern - ${titlePrefix}`;
                    droughtContainer.querySelector('.map-title h4').textContent = `Drought Percentage Pattern - ${titlePrefix}`;
                    
                    // Update the images
                    consistencyContainer.querySelector('img').src = monthPairData.consistency[monthPair];
                    droughtContainer.querySelector('img').src = monthPairData.drought_pct[monthPair];
                    
                    // Update the links
                    consistencyContainer.querySelector('a').href = monthPairData.consistency[monthPair];
                    droughtContainer.querySelector('a').href = monthPairData.drought_pct[monthPair];
                }

                // Year selector
                document.getElementById('yearSelector').addEventListener('change', function() {
                    const selectedYear = this.value;
                    if (selectedYear && yearData[selectedYear]) {
                        const yearInfo = yearData[selectedYear];
                        
                        // Update the combined map
                        document.getElementById('yearlyMapContainer').querySelector('.map-title h4').textContent = `Drought Status for ${selectedYear}`;
                        document.getElementById('yearlyMapImage').innerHTML = `<img src="${yearInfo.combined}" class="img-fluid" alt="Combined map for ${selectedYear}">`;
                        
                        // Update the individual months
                        const monthsContainer = document.getElementById('monthlyMapsRow');
                        monthsContainer.innerHTML = '';
                        
                        yearInfo.months.forEach(month => {
                            const monthDiv = document.createElement('div');
                            monthDiv.className = 'col-md-4 mb-4';
                            monthDiv.innerHTML = `
                                <div class="map-container">
                                    <div class="map-title">
                                        <h5>${month.month} ${selectedYear} (${month.droughtPercent}% Drought)</h5>
                                    </div>
                                    <div class="map-image text-center">
                                        <img src="${month.path}" class="img-fluid" alt="${month.month} ${selectedYear}">
                                    </div>
                                </div>
                            `;
                            monthsContainer.appendChild(monthDiv);
                        });
                    }
                });

                // Map view toggle
                document.getElementById('combinedMapBtn').addEventListener('click', function() {
                    this.classList.add('active');
                    document.getElementById('individualMapBtn').classList.remove('active');
                    document.getElementById('yearlyMapContainer').style.display = 'block';
                    document.getElementById('individualMonthsContainer').style.display = 'none';
                });

                document.getElementById('individualMapBtn').addEventListener('click', function() {
                    this.classList.add('active');
                    document.getElementById('combinedMapBtn').classList.remove('active');
                    document.getElementById('yearlyMapContainer').style.display = 'none';
                    document.getElementById('individualMonthsContainer').style.display = 'block';
                });
            });
        </script>
    </body>
    </html>
    """
    
    # Write the HTML to file
    with open(output_file, 'w') as f:
        f.write(html)
    
    logger.info(f"Interactive dashboard saved to {output_file}")
    write_history(f"Interactive dashboard saved to {output_file}")
    
    return output_file

def create_summary_file(stats, original_file, output_file, drought_threshold, maps, yearly_results, dashboard_file, summary_file="summary.txt"):
    """
    Create a summary text file with details about the calculation and results.
    
    Args:
        stats (dict): Statistics dictionary
        original_file (str): Path to the original CDI file
        output_file (str): Path to the output CDI file
        drought_threshold (float): Threshold used for drought classification
        maps (dict): Dictionary of generated maps
        yearly_results (dict): Results from the yearly maps generation
        dashboard_file (str): Path to the HTML dashboard
        summary_file (str): Path to save the summary file
        
    Returns:
        str: Path to the summary file
    """
    logger.info(f"Step 8: Creating summary file: {summary_file}")
    write_history(f"\nSTEP 8: Creating summary file: {summary_file}")
    
    with open(summary_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("ENHANCED DROUGHT OUTLOOK CDI CALCULATION SUMMARY\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("CALCULATION DETAILS:\n")
        f.write("-" * 50 + "\n")
        f.write(f"Date processed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Original CDI file: {original_file}\n")
        f.write(f"Output file: {output_file}\n")
        f.write(f"Drought threshold: {drought_threshold}\n")
        f.write(f"Time range: {stats['start_time'].strftime('%B %Y')} to {stats['end_time'].strftime('%B %Y')}\n")
        f.write(f"Interactive dashboard: {dashboard_file}\n\n")
        
        f.write("PARALLEL PROCESSING:\n")
        f.write("-" * 50 + "\n")
        f.write(f"CPU cores available: {mp.cpu_count()}\n")
        f.write(f"Parallel processing used for:\n")
        f.write(f"  - Historical consistency calculations\n")
        f.write(f"  - Map generation\n")
        f.write(f"  - Yearly data processing\n\n")
        
        f.write("CALCULATION METHOD:\n")
        f.write("-" * 50 + "\n")
        f.write("Outlook CDI represents the historical consistency of drought status for each month pair:\n")
        f.write(f"  - Drought defined as: CDI ≤ {drought_threshold}\n")
        f.write("  - For each month pair (e.g., April-May):\n")
        f.write("    * Analyze all years (e.g., Apr 1998-May 1998, Apr 1999-May 1999, etc.)\n")
        f.write("    * For each year, value = 1 if both months have same status, 0 if different\n")
        f.write("    * Calculate percentage: (sum of consistent years / total years) * 100\n")
        f.write("    * April-May has 21 years of data (1998-2018)\n")
        f.write("    * January-February has 20 years of data (1999-2018)\n\n")
        
        f.write("ENHANCED FEATURES:\n")
        f.write("-" * 50 + "\n")
        f.write("This enhanced version includes several improvements:\n")
        f.write("  1. Multi-core processing for faster calculations\n")
        f.write("  2. Dual visualization modes (consistency and drought percentage)\n")
        f.write("  3. Interactive HTML dashboard with Bootstrap UI\n")
        f.write("  4. Improved color mapping for better interpretation\n")
        f.write("  5. Yearly collections with drought percentage labeling\n\n")
        
        f.write("MONTH PAIR DETAILS:\n")
        f.write("-" * 50 + "\n")
        f.write(f"{'Month Pair':<15} {'Years':<8} {'Consistency':<25} {'Drought Pct':<25}\n")
        f.write(f"{'':15} {'':8} {'Min %':<8} {'Max %':<8} {'Mean %':<8} {'Min %':<8} {'Max %':<8} {'Mean %':<8}\n")
        f.write("-" * 70 + "\n")
        
        # Sort month pairs in calendar order
        month_pairs = []
        for pair_name in stats['month_pair_stats']:
            month1, month2 = pair_name.split('-')
            month1_idx = list(calendar.month_name).index(month1)
            month_pairs.append((month1_idx, pair_name))
        
        month_pairs.sort()
        
        for _, pair_name in month_pairs:
            pair_stats = stats['month_pair_stats'][pair_name]
            f.write(f"{pair_name:<15} {pair_stats['num_years']:<8} "
                    f"{pair_stats['min_percent']:<8.2f} {pair_stats['max_percent']:<8.2f} {pair_stats['mean_percent']:<8.2f} "
                    f"{pair_stats['min_drought_pct']:<8.2f} {pair_stats['max_drought_pct']:<8.2f} {pair_stats['mean_drought_pct']:<8.2f}\n")
        
        f.write("\n")
        
        # Maps details
        f.write("GENERATED MAPS:\n")
        f.write("-" * 50 + "\n")
        f.write("Maps are organized into categories:\n")
        f.write(f"  1. Month Pair Maps: {len(maps['month_pair_maps']['consistency']) + len(maps['month_pair_maps']['drought_pct'])} total\n")
        f.write(f"     - Consistency maps: {len(maps['month_pair_maps']['consistency'])} maps\n")
        f.write(f"     - Drought percentage maps: {len(maps['month_pair_maps']['drought_pct'])} maps\n")
        f.write(f"  2. Monthly Maps: {len(maps['monthly_maps']['consistency']) + len(maps['monthly_maps']['drought_pct'])} total\n")
        f.write(f"     - Consistency maps: {len(maps['monthly_maps']['consistency'])} maps\n")
        f.write(f"     - Drought percentage maps: {len(maps['monthly_maps']['drought_pct'])} maps\n")
        f.write(f"  3. Yearly Maps: {yearly_results['total_maps']} total maps across {len(yearly_results['years_processed'])} years\n")
        f.write(f"     - Each year has individual month maps and a combined map\n")
        f.write(f"     - Years included: {', '.join(map(str, sorted(yearly_results['years_processed'])))}\n\n")
        
        f.write("INTERACTIVE DASHBOARD:\n")
        f.write("-" * 50 + "\n")
        f.write(f"An interactive HTML dashboard has been created at: {dashboard_file}\n")
        f.write("Features of the dashboard include:\n")
        f.write("  - Toggle between consistency and drought percentage views\n")
        f.write("  - Year selection for viewing yearly drought patterns\n")
        f.write("  - Monthly view options (combined or individual months)\n")
        f.write("  - Seasonal pattern analysis\n")
        f.write("  - Responsive design with Bootstrap UI\n")
        f.write("  - Interactive controls and information modal\n\n")
        
        f.write("USAGE NOTES:\n")
        f.write("-" * 50 + "\n")
        f.write("1. Open the interactive dashboard in a web browser for the best experience\n")
        f.write("2. The NetCDF output file includes both consistency and drought percentage variables\n")
        f.write("3. For programmatic access, use the following variables in the NetCDF file:\n")
        f.write("   - 'outlook_cdi': Historical consistency percentage\n")
        f.write("   - 'drought_pct': Historical drought percentage\n")
        f.write("4. For custom visualizations, use the individual map files in the map directories\n")
        f.write("5. The yearly collections provide a chronological view of drought evolution\n\n")
        
        f.write("RECOMMENDATIONS:\n")
        f.write("-" * 50 + "\n")
        f.write("Based on the analysis, consider the following for drought monitoring:\n")
        f.write("1. Areas with high consistency (>70%) are likely to maintain their drought status\n")
        f.write("2. Areas with high drought percentage (>60%) are historically drought-prone\n")
        f.write("3. Compare recent patterns with historical consistency to identify anomalies\n")
        f.write("4. Use the seasonal patterns tab to understand seasonal drought dynamics\n\n")
        
        f.write("=" * 80 + "\n")
        f.write("End of Summary\n")
        f.write("=" * 80 + "\n")

        logger.info(f"  Summary file created: {summary_file}")
    write_history(f"Summary file created: {summary_file}")
    return summary_file

def main():
    """Main function to run the complete process with parallel processing."""
    start_time = datetime.now()
    logger.info("=" * 80)
    logger.info("ENHANCED DROUGHT OUTLOOK CDI WITH PARALLEL PROCESSING")
    logger.info("=" * 80)
    write_history("=" * 80)
    write_history("STARTING ENHANCED DROUGHT OUTLOOK CDI CALCULATION")
    write_history("Using multi-core processing for improved performance")
    write_history("=" * 80)
    
    # Configuration
    config = {
        'cdi_file': "/Volumes/data/nacp/results/netcdf/cdi_1.nc",  # Original CDI file
        'output_file': "outlook_cdi_2018_enhanced.nc",  # Output file
        'summary_file': "enhanced_summary.txt",  # Summary file
        'history_file': "calculation_history.log",  # History log file
        'maps_dir': "maps",  # Directory to save month-pair maps
        'monthly_maps_dir': "monthly_maps",  # Directory to save monthly outlook maps
        'yearly_maps_dir': "yearly_maps",  # Directory to save yearly map collections
        'dashboard_file': "drought_dashboard.html",  # Interactive dashboard file
        'max_monthly_maps': 50,  # Maximum number of monthly maps to generate (None = all)
        'drought_threshold': 0.3,  # Threshold for drought classification
        'end_year': 2018,   # End year for analysis
        'num_processes': None  # Number of processes to use (None = all available)
    }
    
    # Get number of available CPU cores
    available_cores = mp.cpu_count()
    logger.info(f"Available CPU cores: {available_cores}")
    logger.info(f"Using {'all available' if config['num_processes'] is None else config['num_processes']} cores")
    write_history(f"Available CPU cores: {available_cores}")
    write_history(f"Using {'all available' if config['num_processes'] is None else config['num_processes']} cores")
    
    try:
        # Step 1: Load original CDI file and filter to end year
        logger.info("Step 1: Loading and filtering CDI data...")
        ds_filtered = load_original_cdi(config['cdi_file'], config['end_year'])
        
        # Step 2: Calculate historical consistency using parallel processing
        logger.info("Step 2: Calculating historical consistency in parallel...")
        ds_outlook, stats = calculate_historical_consistency_parallel(
            ds_filtered,
            drought_threshold=config['drought_threshold'],
            num_processes=config['num_processes']
        )
        
        # Step 3: Save outlook CDI to NetCDF file
        logger.info("Step 3: Saving output to NetCDF...")
        output_file = save_outlook_cdi(ds_outlook, config['output_file'])
        
        # Step 4: Generate month-pair maps in parallel
        logger.info("Step 4: Generating month-pair maps...")
        month_pair_maps = generate_maps_parallel(stats, config['maps_dir'])
        
        # Step 5: Generate monthly outlook maps in parallel
        logger.info("Step 5: Generating monthly maps...")
        monthly_maps = generate_monthly_maps_parallel(
            ds_outlook, 
            config['monthly_maps_dir'],
            config['max_monthly_maps'],
            config['num_processes']
        )
        
        # Step 6: Generate yearly map collections in parallel
        logger.info("Step 6: Generating yearly map collections...")
        yearly_results = generate_yearly_maps_parallel(
            ds_filtered,
            config['yearly_maps_dir'],
            config['drought_threshold'],
            config['num_processes']
        )
        
        # Combine all maps for the summary
        all_maps = {
            'month_pair_maps': month_pair_maps,
            'monthly_maps': monthly_maps,
            'yearly_results': yearly_results
        }
        
        # Step 7: Create interactive HTML dashboard
        logger.info("Step 7: Creating interactive dashboard...")
        dashboard_file = create_interactive_dashboard(
            yearly_results,
            month_pair_maps,
            monthly_maps,
            config['dashboard_file']
        )
        
        # Step 8: Create summary file
        logger.info("Step 8: Creating summary file...")
        summary_file = create_summary_file(
            stats,
            config['cdi_file'],
            output_file,
            config['drought_threshold'],
            all_maps,
            yearly_results,
            dashboard_file,
            config['summary_file']
        )
        
        # Log completion
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        logger.info("=" * 80)
        logger.info(f"PROCESS COMPLETED IN {duration:.1f} SECONDS")
        logger.info("=" * 80)
        logger.info(f"Output file: {output_file}")
        logger.info(f"Interactive dashboard: {dashboard_file}")
        logger.info(f"Summary file: {summary_file}")
        logger.info(f"History log: {config['history_file']}")
        logger.info("=" * 80)
        
        write_history("=" * 80)
        write_history(f"PROCESS COMPLETED IN {duration:.1f} SECONDS")
        write_history("=" * 80)
        write_history(f"Output file: {output_file}")
        write_history(f"Interactive dashboard: {dashboard_file}")
        write_history(f"Summary file: {summary_file}")
        write_history("=" * 80)
        
    except Exception as e:
        logger.error(f"Error occurred: {str(e)}")
        write_history(f"ERROR: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        write_history(traceback.format_exc())
    finally:
        # Close the history log file
        history_log.close()

if __name__ == "__main__":
    main()

2025-05-21 15:13:32 [INFO] ================================================================================
2025-05-21 15:13:32 [INFO] ENHANCED DROUGHT OUTLOOK CDI WITH PARALLEL PROCESSING
2025-05-21 15:13:32 [INFO] ================================================================================
2025-05-21 15:13:32 [INFO] Available CPU cores: 8
2025-05-21 15:13:32 [INFO] Using all available cores
2025-05-21 15:13:32 [INFO] Step 1: Loading and filtering CDI data...
2025-05-21 15:13:32 [INFO] Step 1: Loading original CDI file from /Volumes/data/nacp/results/netcdf/cdi_1.nc
2025-05-21 15:13:33 [INFO]   Original dimensions: Frozen({'latitude': 681, 'longitude': 841, 'time': 319})
2025-05-21 15:13:33 [INFO]   Original time range: 1998-04-01 00:00:00 to 2025-04-01 00:00:00
2025-05-21 15:13:33 [INFO]   Filtering to data up to 2018
2025-05-21 15:13:33 [INFO]   Filtered time range: 1998-04-01 00:00:00 to 2018-12-01 00:00:00
2025-05-21 15:13:33 [INFO]   Filtered time steps: 249
2025-05-21 15:13:

Processing month pairs:   0%|          | 0/12 [00:00<?, ?it/s]Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/Users/sabinmaharjan/.pyenv/versions/3.10.13/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/sabinmaharjan/.pyenv/versions/3.10.13/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/sabinmaharjan/.pyenv/versions/3.10.13/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/sabinmaharjan/.pyenv/versions/3.10.13/lib/python3.10/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'process_month_pair' on <module '__main__' (built-in)>
Process SpawnPoolWorker-2:
Traceback (most recent call last):
  File "/Users/sabinmaharjan/.pyenv/versions/3.10.13/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/

KeyboardInterrupt: 